# Обнаружение точек разладки

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Обнаружение точек разладки».

## Что делает метод

Точка разладки — момент во временном ряду, после которого статистические свойства процесса резко меняются: смещается среднее значение, изменяется разброс или характер колебаний. Обнаружение разладок отвечает на вопрос «когда процесс сменил режим»: начало эпидемии, отказ оборудования, смена климатического тренда, реакция системы на вмешательство.

В этом блокноте мы реализуем поиск разладок методом минимизации стоимости сегментации (динамическое программирование). Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы смотрите на кардиограмму или сейсмограф: большую часть времени кривая «спокойная», и вдруг характер колебаний резко меняется — амплитуда, средний уровень или ритм становятся другими. Именно этот момент перелома и называется точкой разладки.

Идея метода проста: попробуем разбить весь ряд на участки так, чтобы внутри каждого участка данные были как можно более однородными (похожими друг на друга), а различия концентрировались на границах между участками. Чем меньше «разброс» внутри сегмента относительно его среднего, тем лучше — это и есть стоимость сегментации.

Метод перебирает разные варианты разбиений и находит оптимальный — тот, при котором суммарная стоимость всех сегментов минимальна. Чтобы алгоритм не дробил ряд на бесконечно мелкие однородные куски, за каждую дополнительную границу берётся **штраф** — плата за сложность. Баланс между качеством разбиения и штрафом управляет количеством найденных точек разладки.

**Точка разладки** — момент, после которого статистические свойства процесса (среднее, разброс, характер колебаний) резко меняются. **Сегмент** — участок ряда между двумя соседними точками разладки, внутри которого процесс считается однородным.

## 1. Установка библиотек

In [ ]:
%pip install -q numpy==1.26.4 pandas==2.2.2 matplotlib==3.9.2

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сформируем синтетический ряд из четырёх участков с намеренно разным средним уровнем и разным разбросом (стандартным отклонением). Это имитирует, например, прибор, переключающийся между режимами, или климатический ряд с несколькими фазами.

Истинные точки разладки нам известны заранее: по ним в конце проверим, насколько точно алгоритм их нашёл.

Запустите ячейку — увидите длину ряда и позиции (номера наблюдений) истинных разладок.

In [ ]:
import numpy as np
import pandas as pd

# Фиксируем зерно для воспроизводимости
rng = np.random.default_rng(42)

# Каждый сегмент: (длина, среднее, стандартное отклонение).
segments = [(80, 5.0, 0.6), (60, 8.5, 0.6), (90, 8.0, 1.8), (70, 3.0, 0.7)]
parts, true_cp = [], []
pos = 0
for length, mean, std in segments:
    parts.append(rng.normal(mean, std, length))
    pos += length
    true_cp.append(pos)
true_cp = true_cp[:-1]              # последняя граница - конец ряда

signal = np.concatenate(parts)
print(f"Длина ряда: {len(signal)} наблюдений")
print(f"Истинные точки разладки: {true_cp}")

## 4. Применение метода

### Шаг 1. Функция стоимости сегмента

Нам нужно измерить, насколько «однороден» участок ряда. В качестве меры используем сумму квадратов отклонений каждого значения от среднего по этому участку — чем меньше это число, тем однороднее сегмент.

Эта ячейка определяет вспомогательную функцию; никаких выводов на экран она не даёт.

In [ ]:
def segment_cost(x):
    """Стоимость сегмента: разброс относительно собственного среднего."""
    if len(x) == 0:
        return 0.0
    return float(np.sum((x - x.mean()) ** 2))

### Шаг 2. Поиск оптимального разбиения динамическим программированием

**Динамическое программирование** — алгоритмическая техника, позволяющая найти оптимальное решение, не перебирая все варианты явно. Для задачи сегментации это означает: чтобы найти лучшее разбиение ряда длиной N, достаточно перебрать, где поставить последнюю границу, а стоимость всего, что слева, уже вычислена ранее.

**Штраф `penalty`** — ключевой параметр: за каждую новую точку разладки алгоритм «платит» этот штраф. Если штраф мал — алгоритм найдёт много разладок, включая ложные от шума. Если штраф велик — пропустит реальные слабые сдвиги. Подбор штрафа — это главное методическое решение.

После запуска ячейки на экране появятся найденные и истинные позиции точек разладки — сравните их.

In [ ]:
def detect_change_points(x, penalty):
    """Возвращает оптимальный набор точек разладки.

    penalty - штраф за каждую дополнительную точку: больше штраф -
    меньше разладок.
    """
    n = len(x)
    # best[i] - минимальная стоимость разбиения первых i точек.
    best = np.full(n + 1, np.inf)
    best[0] = -penalty
    prev = np.zeros(n + 1, dtype=int)
    for end in range(1, n + 1):
        for start in range(0, end):
            cost = best[start] + segment_cost(x[start:end]) + penalty
            if cost < best[end]:
                best[end] = cost
                prev[end] = start
    # Восстанавливаем границы сегментов обратным ходом.
    points, end = [], n
    while end > 0:
        start = prev[end]
        if start > 0:
            points.append(start)
        end = start
    return sorted(points)


penalty = 12.0
found_cp = detect_change_points(signal, penalty)
print(f"Найденные точки разладки: {found_cp}")
print(f"Истинные точки разладки:  {true_cp}")

### Шаг 3. Визуализация найденных разладок

Строим итоговый график: ряд, горизонтальные линии среднего уровня внутри каждого найденного сегмента, истинные и найденные точки разладки. По нему сразу видно, насколько алгоритм угадал реальные переломы.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(13, 5.4))
ax.plot(signal, color=VIZ["series"][3], linewidth=1.4, label="Ряд")

# Средний уровень внутри каждого найденного сегмента.
bounds = [0] + found_cp + [len(signal)]
for s, e in zip(bounds[:-1], bounds[1:]):
    ax.hlines(signal[s:e].mean(), s, e, color=VIZ["series"][0],
              linewidth=2.6)
ax.plot([], [], color=VIZ["series"][0], linewidth=2.6,
        label="Средний уровень сегмента")

for cp in true_cp:
    ax.axvline(cp, color=VIZ["series"][2], linestyle="--", linewidth=1.6)
ax.plot([], [], color=VIZ["series"][2], linestyle="--",
        label="Истинная разладка")
for cp in found_cp:
    ax.axvline(cp, color=VIZ["series"][1], linestyle=":", linewidth=2.0)
ax.plot([], [], color=VIZ["series"][1], linestyle=":",
        label="Найденная разладка")

ax.set_title("Обнаружение точек разладки во временном ряду")
ax.set_xlabel("Номер наблюдения")
ax.set_ylabel("Значение")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Как читать этот график.**
- Серо-синяя кривая — сам ряд со всем шумом.
- Синие горизонтальные отрезки — средний уровень каждого найденного сегмента. Если сегмент выделен правильно, отрезок проходит «по центру» своего участка ряда.
- Оранжевые пунктирные вертикали — истинные точки разладки (знаем заранее, потому что данные синтетические).
- Зелёные пунктирные вертикали — найденные точки разладки.
- Хороший результат: зелёные линии близки к оранжевым. Точное совпадение до одного наблюдения при наличии шума не обязательно; важна правильная локализация.

## 5. Интерпретация результата

- **Найденные точки** делят ряд на однородные сегменты; горизонтальные линии — средний уровень каждого сегмента. На графике найденные разладки должны почти совпадать с истинными.
- **Штраф `penalty`** управляет чувствительностью: малый штраф находит много разладок (включая ложные от шума), большой штраф пропускает слабые изменения. Это главный параметр метода, его подбирают под задачу.
- Метод этой версии реагирует прежде всего на сдвиг среднего. Изменение только разброса (как между третьим и соседними сегментами) улавливается хуже; для него нужна функция стоимости, учитывающая дисперсию.
- Близко расположенные истинная и найденная границы — это успех: точное совпадение «номер в номер» при наличии шума не гарантируется.

Помните: метод указывает, *где* сменился режим, но не объясняет *почему*. Причину устанавливает исследователь.

## 6. Поэкспериментируйте сами

Поменяйте один параметр, повторно запустите ячейки раздела 4 и посмотрите, что изменится.

**Что менять и что наблюдать:**
- Уменьшите `penalty` до 3.0. Алгоритм найдёт больше разладок — появятся ложные. Это называется переобнаружением.
- Увеличьте `penalty` до 50.0. Алгоритм найдёт меньше разладок или не найдёт ни одной. Это пропуск реальных сдвигов.
- Увеличьте шум в третьем сегменте: измените `(90, 8.0, 1.8)` на `(90, 8.0, 5.0)`. Станет ли сложнее найти соседние границы?
- Измените среднее одного сегмента так, чтобы оно совпало с соседним (например, `(60, 5.1, 0.6)` вместо `(60, 8.5, 0.6)`). Сможет ли алгоритм найти эту «незаметную» разладку?

In [ ]:
# --- Шаблон загрузки своих данных ---
# raw = pd.read_csv("my_signal.csv")
# signal = raw["значение"].astype(float).to_numpy()
#
# penalty = 20.0                         # подберите под свой ряд
# found_cp = detect_change_points(signal, penalty)
# print(f"Найденные точки разладки: {found_cp}")
# Затем повторите ячейку визуализации (без линий истинных разладок).

## Краткие выводы

- Точка разладки — момент, когда статистические свойства ряда (среднее, разброс) резко меняются. Метод находит такие моменты, минимизируя неоднородность внутри сегментов.
- Штраф `penalty` — главный параметр: он определяет чувствительность. Малый штраф — много ложных разладок; большой штраф — пропуск реальных.
- Метод в этой реализации реагирует на сдвиг среднего. Для обнаружения изменений разброса или формы распределения нужна другая функция стоимости.
- Алгоритм указывает, *когда* сменился режим, но не объясняет *почему* — причину устанавливает исследователь, опираясь на предметные знания.
- На синтетических данных с известными разладками удобно калибровать штраф перед применением к реальным данным.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На графике найденная точка разладки отстоит от истинной на несколько наблюдений, хотя не совпадает с ней точно. Является ли это ошибкой алгоритма или нормальным результатом, и как оценивать качество локализации при наличии шума?
2. Третий сегмент данных (позиции 140–230) отличается от соседних повышенным разбросом, а не сдвигом среднего. Почему функция стоимости `segment_cost`, используемая в блокноте, может хуже улавливать этот тип разладки, чем сдвиг среднего?
3. Исследователь применяет алгоритм к реальному ряду и выбирает штраф `penalty`, ориентируясь на число найденных разладок: выбирает то значение, при котором находится «разумное» число точек. Какой методологический риск несёт такой подход, и что является более строгой альтернативой?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Небольшое смещение оценки разладки — нормальный результат в присутствии шума. Истинная точка разладки статистически не определена с точностью до одного наблюдения: алгоритм находит оптимальное разбиение по имеющимся данным, и отклонение порядка одного-двух стандартных отклонений ряда считается приемлемым. Качество локализации принято оценивать как среднее абсолютное отклонение от истинных точек на нескольких синтетических реализациях.
2. Функция стоимости — сумма квадратов отклонений от среднего — минимизируется, когда среднее сегмента хорошо описывает данные. При изменении только дисперсии среднее двух соседних участков с одинаковым уровнем, но разным разбросом совпадает, и стоимость единого сегмента оказывается ненамного хуже двух раздельных. Для детекции изменений дисперсии нужна функция стоимости, явно учитывающая логарифм правдоподобия нормального распределения.
3. Выбор штрафа по «разумности» числа разладок — это подбор под ожидаемый результат, что создаёт риск подтверждения предубеждений. Более строгая альтернатива — критерии информационной сложности (например, BIC или MBIC), которые выводят штраф аналитически из размера выборки и числа параметров модели, не опираясь на субъективные представления о «разумном» числе точек.
</details>
